In [2]:
import pandas as pd

In [3]:
get_decoder = lambda x: [i["config"].split("_")[0] for i in x.to_dict(orient="records")]
if_cfenet = lambda x: ["cfenet" in i["config"] for i in x.to_dict(orient="records")]
dataset_map = {
    "mas": "Massachusetts",
    "whu": "WHU",
}

In [4]:
get_training_dataset = lambda x: [dataset_map.get(i["config"].split("_")[1]) for i in x.to_dict(orient="records")]

In [13]:
def assign_name(name):
    map_names = {
        "cfenet": "CFENet",
        "unet": "UNet",
        "upernet": "UPerNet",
        "dlab": "DeepLabV3+"
    }

    name_parts = name.split("_")
    if_cfenet = "cfenet" in name_parts
    decoder = name_parts[0]
    return " + ".join([map_names[decoder], "CFENet"]) if if_cfenet else map_names[decoder]

def preprocess_benchmarks(benchmarks):
    benchmarks["decoder"] = get_decoder(benchmarks)
    benchmarks["cfenet"] = if_cfenet(benchmarks)
    benchmarks["training_dataset"] = get_training_dataset(benchmarks)
    benchmarks["name"] = [assign_name(name) for name in benchmarks["config"].to_list()]
    return benchmarks

In [23]:
def present_benchmarks(benchmarks):
    b =  benchmarks.iloc[:, [-1, -2, 1, 3, 4, 5, 6, 7, 8, 9, 16, 17]]
    # set columns name and training_dataset as index
    # b = b.set_index(["name", "training_dataset"])
    return b



In [21]:
benchmarks_42 = pd.read_csv("final_42_benchmarks.csv")
preprocess_benchmarks(benchmarks_42)
benchmarks_42.head()

,config,dataset,timestamp,pos_iou,precision,recall,f1,accuracy,neg_iou,mean_iou,...,fn,tn,intersection,union,boundary_iou,threshold,decoder,cfenet,training_dataset,name
0,dlab_mas_main_best,WHU Test,24-07-2026 21:40:00,0.747709,0.819091,0.895613,0.855644,0.953097,0.945525,0.846617,...,5215464,262067022,35352892,52022567,0.679568,0.42,dlab,False,Massachusetts,DeepLabV3+
1,dlab_mas_main_best,Massachusetts,24-07-2026 21:40:00,0.755426,0.851446,0.870108,0.860675,0.947585,0.937460,0.846443,...,543792,17677960,3615595,4806079,0.752296,0.42,dlab,False,Massachusetts,DeepLabV3+
2,unet_mas_final_best,WHU Test,24-07-2026 22:23:22,0.711511,0.822797,0.840271,0.831442,0.947122,0.939192,0.825352,...,7980469,262908534,32484898,50973485,0.637290,0.44,unet,False,Massachusetts,UNet
3,unet_mas_final_best,Massachusetts,24-07-2026 22:23:22,0.736402,0.829974,0.867230,0.848193,0.942240,0.931126,0.833764,...,555839,17569747,3599537,4905375,0.733794,0.44,unet,False,Massachusetts,UNet
4,dlab_whu_main_best,WHU Test,24-07-2026 23:02:07,0.897705,0.950570,0.941663,0.946095,0.983346,0.980495,0.939100,...,2914692,269503635,38651609,44871656,0.861381,0.50,dlab,False,WHU,DeepLabV3+


In [15]:
inria_benchmarks_42 = preprocess_benchmarks(pd.read_csv("final_42_inria_benchmarks.csv"))
inria_benchmarks_42.head()

,config,dataset,timestamp,pos_iou,precision,recall,f1,accuracy,neg_iou,mean_iou,...,fn,tn,intersection,union,boundary_iou,threshold,decoder,cfenet,training_dataset,name
0,dlab_mas_main_best,austin,24-07-2026 22:01:30,0.550385,0.729179,0.691801,0.709997,0.924608,0.916942,0.733664,...,5139425,104039734,11536219,20960266,0.550385,0.42,dlab,False,Massachusetts,DeepLabV3+
1,dlab_mas_main_best,chicago,24-07-2026 22:01:30,0.573688,0.676538,0.790516,0.729099,0.880986,0.858290,0.715989,...,5305120,90103578,19979488,34854530,0.573225,0.42,dlab,False,Massachusetts,DeepLabV3+
2,dlab_mas_main_best,kitsap,24-07-2026 22:01:30,0.366608,0.452302,0.659283,0.536522,0.977147,0.976840,0.671724,...,854500,120489867,1653450,4510133,0.366608,0.42,dlab,False,Massachusetts,DeepLabV3+
3,dlab_mas_main_best,tyrol-w,24-07-2026 22:01:30,0.622196,0.737889,0.798726,0.767103,0.964851,0.962691,0.792443,...,1823376,113370521,7235814,11629479,0.622196,0.42,dlab,False,Massachusetts,DeepLabV3+
4,dlab_mas_main_best,vienna,24-07-2026 22:01:30,0.627974,0.799322,0.745511,0.771479,0.884700,0.856837,0.742405,...,8304694,86259425,24199232,38660411,0.625943,0.42,dlab,False,Massachusetts,DeepLabV3+


In [36]:
benchmarks_123 = preprocess_benchmarks(pd.read_csv("final_123_benchmarks.csv"))
inria_benchmarks_123 = preprocess_benchmarks(pd.read_csv("final_123_inria_benchmarks.csv"))
benchmarks_456 = preprocess_benchmarks(pd.read_csv("final_456_benchmarks.csv"))
inria_benchmarks_456 = preprocess_benchmarks(pd.read_csv("final_456_inria_benchmarks.csv"))


In [37]:
present_42 = present_benchmarks(benchmarks_42)
present_123 = present_benchmarks(benchmarks_123)
present_456 = present_benchmarks(benchmarks_456)
inria_present_42 = present_benchmarks(inria_benchmarks_42)
inria_present_123 = present_benchmarks(inria_benchmarks_123)
inria_present_456 = present_benchmarks(inria_benchmarks_456)


In [38]:
import pandas as pd
import numpy as np

def summarize_seeds(df1, df2, df3,
                    group_cols=("name", "training_dataset", "dataset"),
                    decimals=4):
    """
    Combine results from three seeds into a dataframe of mean ± std.

    Parameters
    ----------
    df1, df2, df3 : pd.DataFrame
        DataFrames for the three seeds.

    group_cols : tuple
        Columns uniquely identifying an experiment.

    decimals : int
        Number of decimal places.

    Returns
    -------
    pd.DataFrame
        Same rows as the input experiments, with every numeric metric
        formatted as 'mean ± std'.
    """

    df = pd.concat([df1, df2, df3], ignore_index=True)

    metric_cols = [
        c for c in df.columns
        if c not in group_cols and np.issubdtype(df[c].dtype, np.number)
    ]

    grouped = (
        df.groupby(list(group_cols))[metric_cols]
          .agg(["mean", "std"])
    )

    result = pd.DataFrame(index=grouped.index)

    for metric in metric_cols:
        mean = grouped[(metric, "mean")]
        std = grouped[(metric, "std")].fillna(0)

        result[metric] = (
            mean.map(lambda x: f"{x:.{decimals}f}")
            + " ± "
            + std.map(lambda x: f"{x:.{decimals}f}")
        )

    return result.reset_index()

In [40]:
final_pixel_benchmarks = summarize_seeds(present_42, present_123, present_456)
final_pixel_benchmarks.to_csv("final_benchmarks.csv", index=False)

In [48]:
mas_benchmarks = final_pixel_benchmarks[final_pixel_benchmarks["training_dataset"] == "Massachusetts"]
mas_benchmarks = mas_benchmarks.drop(columns=["training_dataset"],)
mas_benchmarks.to_csv("trained_on_mas_benchmarks.csv", index=False)

In [50]:
whu_benchmarks = final_pixel_benchmarks[final_pixel_benchmarks["training_dataset"] == "WHU"]
whu_benchmarks = whu_benchmarks.drop(columns=["training_dataset"],)
whu_benchmarks.to_csv("trained_on_whu_benchmarks.csv", index=False)

In [51]:
final_inria_benchmarks = summarize_seeds(inria_present_42, inria_present_123, inria_present_456)
final_inria_benchmarks.to_csv("final_inria_benchmarks.csv", index=False)

In [57]:
import pandas as pd

def pivot_city_metrics(df, metric_col):
    """
    Reshape model performance metrics by city.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Must contain columns: 'name', 'training_dataset', 'dataset', and the metric column.
    metric_col : str
        Name of the metric column to pivot (e.g., 'pos_iou', 'precision', 'recall', etc.)
    
    Returns
    -------
    pandas.DataFrame
        Columns: 'name', 'training_dataset', then each city (austin, chicago, kitsap, tyrolw, vienna)
        as separate columns with the corresponding metric values.
        Rows correspond to each unique (name, training_dataset) pair.
        The 'overall' dataset row is excluded.
    """
    # Define the city names (exclude 'overall')
    cities = ['austin', 'chicago', 'kitsap', 'tyrol-w', 'vienna', 'overall']
    
    # Filter to only city rows
    df_cities = df[df['dataset'].isin(cities)].copy()
    
    # Pivot: index = (name, training_dataset), columns = dataset, values = metric_col
    pivoted = df_cities.pivot_table(
        index=['name', 'training_dataset'],
        columns='dataset',
        values=metric_col,
        aggfunc='first'  # in case of duplicates, keeps first; use 'mean' if appropriate
    ).reset_index()
    
    # Ensure city columns are in the desired order
    pivoted = pivoted[['name', 'training_dataset'] + cities]
    
    return pivoted

In [59]:
city_wise_metrics = pivot_city_metrics(final_inria_benchmarks, 'pos_iou')
city_wise_metrics.to_csv("city_wise_metrics.csv", index=False)

In [61]:
city_wise_metrics[city_wise_metrics["training_dataset"] == "Massachusetts"]

dataset,name,training_dataset,austin,chicago,kitsap,tyrol-w,vienna,overall
0,DeepLabV3+,Massachusetts,0.5570 ± 0.0062,0.5665 ± 0.0064,0.4533 ± 0.0751,0.6419 ± 0.0178,0.6422 ± 0.0145,0.5955 ± 0.0095
2,DeepLabV3+ + CFENet,Massachusetts,0.5345 ± 0.0320,0.5769 ± 0.0064,0.3612 ± 0.0336,0.6209 ± 0.0112,0.6493 ± 0.0007,0.5946 ± 0.0080
4,UNet,Massachusetts,0.5332 ± 0.0322,0.5643 ± 0.0085,0.3722 ± 0.0808,0.6323 ± 0.0162,0.6232 ± 0.0043,0.5791 ± 0.0128
6,UNet + CFENet,Massachusetts,0.4826 ± 0.0163,0.5633 ± 0.0121,0.2880 ± 0.0301,0.5715 ± 0.0242,0.6177 ± 0.0270,0.5590 ± 0.0112
8,UPerNet,Massachusetts,0.5414 ± 0.0121,0.5677 ± 0.0007,0.3578 ± 0.0290,0.6470 ± 0.0217,0.6057 ± 0.0293,0.5770 ± 0.0112
10,UPerNet + CFENet,Massachusetts,0.5318 ± 0.0389,0.5703 ± 0.0177,0.4055 ± 0.0515,0.6361 ± 0.0062,0.6188 ± 0.0126,0.5832 ± 0.0167


In [62]:
city_wise_metrics[city_wise_metrics["training_dataset"] == "WHU"]

dataset,name,training_dataset,austin,chicago,kitsap,tyrol-w,vienna,overall
1,DeepLabV3+,WHU,0.6052 ± 0.0480,0.6089 ± 0.0083,0.4450 ± 0.0704,0.3530 ± 0.0622,0.6432 ± 0.0336,0.5652 ± 0.0267
3,DeepLabV3+ + CFENet,WHU,0.6358 ± 0.0241,0.5966 ± 0.0398,0.6133 ± 0.0652,0.5856 ± 0.0773,0.6370 ± 0.0499,0.6171 ± 0.0403
5,UNet,WHU,0.6207 ± 0.0155,0.6152 ± 0.0124,0.5477 ± 0.0678,0.5667 ± 0.1040,0.6820 ± 0.0119,0.6306 ± 0.0215
7,UNet + CFENet,WHU,0.6560 ± 0.0043,0.6240 ± 0.0047,0.6270 ± 0.0269,0.6620 ± 0.0230,0.6731 ± 0.0034,0.6520 ± 0.0024
9,UPerNet,WHU,0.6445 ± 0.0264,0.6232 ± 0.0039,0.5862 ± 0.0458,0.6323 ± 0.0637,0.6798 ± 0.0091,0.6469 ± 0.0147
11,UPerNet + CFENet,WHU,0.6545 ± 0.0210,0.6124 ± 0.0075,0.6521 ± 0.0084,0.6507 ± 0.0331,0.6702 ± 0.0155,0.6466 ± 0.0139


In [63]:
import pickle

def combine_pickles(pkl1_path, pkl2_path, output_path):
    """
    Combine two pickle files and save the result.

    - list + list -> concatenation
    - dict + dict -> update (second overrides duplicate keys)

    Returns the combined object.
    """
    with open(pkl1_path, "rb") as f:
        obj1 = pickle.load(f)

    with open(pkl2_path, "rb") as f:
        obj2 = pickle.load(f)

    if isinstance(obj1, list) and isinstance(obj2, list):
        combined = obj1 + obj2

    elif isinstance(obj1, dict) and isinstance(obj2, dict):
        combined = obj1.copy()
        combined.update(obj2)

    else:
        raise TypeError(
            f"Unsupported types: {type(obj1)} and {type(obj2)}"
        )

    with open(output_path, "wb") as f:
        pickle.dump(combined, f, protocol=pickle.HIGHEST_PROTOCOL)

    return combined

In [114]:
combine_pickles(
    "instance_metrics_s123.pkl",
    "instance_metrics_s123 (1).pkl",
    "instance metrics_123.pkl"
)

{'unet_mas_main_s123_best': {'WHU Test': {'iou_0.5': {'global': {'tp': 19312,
     'fp': 12854,
     'fn': 11832,
     'gt': 31144,
     'discarded_preds': 5210,
     'precision': 0.600385500217621,
     'recall': 0.6200873362445415,
     'f1': 0.6100773969357131,
     'iou': 0.4389290422291922},
    'size': {'small': {'tp': 6419,
      'fn': 7756,
      'gt': 14175,
      'recall': 0.4528395061728395},
     'medium': {'tp': 11440,
      'fn': 3808,
      'gt': 15248,
      'recall': 0.7502623294858342},
     'large': {'tp': 1127, 'fn': 219, 'gt': 1346, 'recall': 0.837295690936107},
     'very_large': {'tp': 291,
      'fn': 46,
      'gt': 337,
      'recall': 0.8635014836795252},
     'outlier': {'tp': 35, 'fn': 3, 'gt': 38, 'recall': 0.9210526315789473}}},
   'iou_0.3': {'global': {'tp': 21997,
     'fp': 10200,
     'fn': 9147,
     'gt': 31144,
     'discarded_preds': 5179,
     'precision': 0.6832002981644253,
     'recall': 0.7062997688158232,
     'f1': 0.6945580271861829,
    

In [67]:
import pickle

In [123]:
with open("instance metrics_42.pkl", "rb") as f, open("instance metrics_123.pkl", "rb") as f2, open("instance metrics_456.pkl", "rb") as f3:
    im42 = pickle.load(f)
    im123 = pickle.load(f2)
    im456 = pickle.load(f3)
del im42["upernet_whu_cfenet_final_best"], im42["upernet_mas_cfenet_final-2_best"]


In [75]:
get_trn_dataset = lambda x: dataset_map[x.split("_")[1]]

In [76]:
[" - ".join([assign_name(i), get_trn_dataset(i)]) for i in im42.keys()]

['DeepLabV3+ - Massachusetts',
 'UNet - Massachusetts',
 'UPerNet + CFENet - Massachusetts',
 'DeepLabV3+ - WHU',
 'UNet - WHU',
 'UPerNet - Massachusetts',
 'UPerNet - WHU',
 'UPerNet + CFENet - WHU',
 'DeepLabV3+ + CFENet - WHU',
 'DeepLabV3+ + CFENet - Massachusetts',
 'UNet + CFENet - Massachusetts',
 'UNet + CFENet - WHU',
 'UPerNet + CFENet - Massachusetts',
 'UPerNet + CFENet - WHU']

In [136]:
list(im42.items())[0][1]["WHU Test"]["iou_0.5"]

{'global': {'tp': 20398,
  'fp': 13443,
  'fn': 10746,
  'gt': 31144,
  'precision': 0.6027599657220531,
  'recall': 0.6549576162342666,
  'f1': 0.6277756405324306},
 'size': {'small': {'tp': 7229,
   'fn': 6946,
   'gt': 14175,
   'recall': 0.5099823633156967},
  'medium': {'tp': 11707,
   'fn': 3541,
   'gt': 15248,
   'recall': 0.7677728226652676},
  'large': {'tp': 1134, 'fn': 212, 'gt': 1346, 'recall': 0.8424962852897474},
  'very_large': {'tp': 292, 'fn': 45, 'gt': 337, 'recall': 0.8664688427299704},
  'outlier': {'tp': 36, 'fn': 2, 'gt': 38, 'recall': 0.9473684210526315}}}

In [86]:
datasets = ['WHU Test', 'Massachusetts', 'austin', 'chicago', 'kitsap', 'tyrol-w', 'vienna', 'overall']

In [137]:
def get_global_metrics(im):
    l = []
    global_metrics05 = {k: [] for k in datasets}
    global_metrics03 = {k: [] for k in datasets}
    seg_error = {k: [] for k in datasets}
    size_recalls = {k: [] for k in datasets}

    for k, i in im.items():
        name = " - ".join([assign_name(k), get_trn_dataset(k)])
        for d in datasets:
            try:
                g = i[d]["iou_0.5"]["global"]
                global_metrics05[d].append([name, g["precision"], g["recall"], g["f1"],])
                g = i[d]["iou_0.3"]["global"]
                global_metrics03[d].append([name, g["precision"], g["recall"], g["f1"],])
                s = i[d]["seg_error"]
                seg_error[d].append([name, s["under_seg_ratio"], s["over_seg_ratio"], s["average_under_severity"], s["average_over_severity"],])
                sizes = i[d]["iou_0.5"]["size"]
                for s in sizes:
                    size_recalls[d].append([name, str(s), sizes[s]["recall"]])
            except KeyError:
                l.append((k, d))
    return global_metrics05, global_metrics03, seg_error, size_recalls, l

In [138]:
gm_42 = get_global_metrics(im42)
gm_123 = get_global_metrics(im123)
gm_456 = get_global_metrics(im456)

In [121]:
im42.keys()

dict_keys(['dlab_mas_main_best', 'unet_mas_final_best', 'upernet_mas_cfenet_final-2_best', 'dlab_whu_main_best', 'unet_whu_final_best', 'upernet_mas_main_best', 'upernet_whu_main_best', 'upernet_whu_cfenet_final_best', 'dlab_whu_cfenet_main_best', 'dlab_mas_cfenet_final-2_best', 'unet_mas_cfenet_final_best', 'unet_whu_cfenet_tmax100_best', 'upernet_mas_cfenet_main_s42_best', 'upernet_whu_cfenet_main_s42_best'])

In [131]:
gm_123[2]

{'WHU Test': [['UNet - Massachusetts',
   0.07244808788647997,
   0.014941093859717562,
   1.1470419299253303,
   1.0287206266318538],
  ['UNet + CFENet - Massachusetts',
   0.05468313040485307,
   0.021336420774065498,
   1.156569630212431,
   1.0251937984496124],
  ['DeepLabV3+ + CFENet - WHU',
   0.02040398525999727,
   0.012416686402544237,
   1.0568561872909699,
   1.0435967302452316],
  ['UPerNet - WHU',
   0.025636607973536403,
   0.011096029860875467,
   1.0564516129032258,
   1.0305810397553516],
  ['UPerNet + CFENet - WHU',
   0.02252777396790564,
   0.00998003992015968,
   1.060882800608828,
   1.023728813559322],
  ['UNet + CFENet - WHU',
   0.026780175087888606,
   0.008999255700656336,
   1.0527670527670527,
   1.0300751879699248],
  ['UNet - WHU',
   0.028080089298172177,
   0.01106808758523798,
   1.0509316770186334,
   1.0247678018575852],
  ['DeepLabV3+ - WHU',
   0.021725700164744646,
   0.012715898272813818,
   1.0489731437598737,
   1.0374331550802138],
  ['DeepLab

In [132]:
import pandas as pd

def segerr_dict_to_dataframe(results):
    rows = []

    for test_dataset, experiments in results.items():
        for exp in experiments:
            model_train, a, b, c, d = exp

            model, training_dataset = model_train.split(" - ", 1)

            rows.append({
                "name": model,
                "training_dataset": training_dataset,
                "dataset": test_dataset,
                "under_seg_ratio": a,
                "over_seg_ratio": b,
                "avg_under_severity": c,
                "avg_over_severity": d
            })

    return pd.DataFrame(rows)

In [133]:
import pandas as pd

def instance_dict_to_dataframe(results):
    rows = []

    for test_dataset, experiments in results.items():
        for exp in experiments:
            model_train, precision, recall, f1 = exp

            model, training_dataset = model_train.split(" - ", 1)

            rows.append({
                "name": model,
                "training_dataset": training_dataset,
                "dataset": test_dataset,
                "instance_precision": precision,
                "instance_recall": recall,
                "instance_f1": f1,
            })

    return pd.DataFrame(rows)

In [139]:
import pandas as pd

def size_dict_to_dataframe(results):
    rows = []

    for test_dataset, experiments in results.items():
        for exp in experiments:
            model_train, size, recall = exp

            model, training_dataset = model_train.split(" - ", 1)

            rows.append({
                "name": model,
                "training_dataset": training_dataset,
                "dataset": test_dataset,
                "size": size,
                "recall": recall,
            })

    return pd.DataFrame(rows)

In [127]:
imdf42 = instance_dict_to_dataframe(gm_42[0])
imdf123 = instance_dict_to_dataframe(gm_123[0])
imdf456 = instance_dict_to_dataframe(gm_456[0])


In [129]:
summarize_seeds(imdf42, imdf123, imdf456).to_csv("instance_metrics_iou_0.5.csv")

In [135]:
segerr42 = segerr_dict_to_dataframe(gm_42[2])
segerr123 = segerr_dict_to_dataframe(gm_123[2])
segerr456 = segerr_dict_to_dataframe(gm_456[2])
summarize_seeds(segerr42, segerr123, segerr456).to_csv("seg_error_metrics.csv")

In [149]:
size_recalls42 = size_dict_to_dataframe(gm_42[3])
size_recalls123 = size_dict_to_dataframe(gm_123[3])
size_recalls456 = size_dict_to_dataframe(gm_456[3])


In [151]:
import pandas as pd
import numpy as np

def summarize_seeds_size(df1, df2, df3, decimals=4):
    """
    Summarize three seed DataFrames grouped by
    name, training_dataset, dataset, and size.
    """

    group_cols = ["name", "training_dataset", "dataset", "size"]

    df = pd.concat([df1, df2, df3], ignore_index=True)

    metric_cols = [
        c for c in df.columns
        if c not in group_cols and np.issubdtype(df[c].dtype, np.number)
    ]

    grouped = (
        df.groupby(group_cols)[metric_cols]
          .agg(["mean", "std"])
    )

    result = pd.DataFrame(index=grouped.index)

    for metric in metric_cols:
        mean = grouped[(metric, "mean")]
        std = grouped[(metric, "std")].fillna(0)

        result[metric] = (
            mean.map(lambda x: f"{x:.{decimals}f}")
            + " ± "
            + std.map(lambda x: f"{x:.{decimals}f}")
        )

    return result.reset_index()

In [153]:
summarize_seeds_size(size_recalls42, size_recalls123, size_recalls456).to_csv("size_recalls.csv", index=False)

In [148]:
size_recalls456

,name,training_dataset,dataset,size,recall
0,DeepLabV3+,Massachusetts,WHU Test,small,0.531993
1,DeepLabV3+,Massachusetts,WHU Test,medium,0.773413
2,DeepLabV3+,Massachusetts,WHU Test,large,0.829866
3,DeepLabV3+,Massachusetts,WHU Test,very_large,0.824926
4,DeepLabV3+,Massachusetts,WHU Test,outlier,0.947368
...,...,...,...,...,...
475,DeepLabV3+ + CFENet,WHU,overall,small,0.316436
476,DeepLabV3+ + CFENet,WHU,overall,medium,0.433624
477,DeepLabV3+ + CFENet,WHU,overall,large,0.702770
478,DeepLabV3+ + CFENet,WHU,overall,very_large,0.650124
